# 01 · Data Exploration — Dunnhumby Complete Journey

Exploratory analysis of all eight raw files in `data/raw/`. Goals:

- Document available fields and entity counts
- Characterize the price and promotion signals available as causal treatments
- Identify gaps relevant to causal pricing estimation
- Record the decision on whether a simulation layer is needed

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

REPO_ROOT = Path("..").resolve()
DATA_RAW  = REPO_ROOT / "data" / "raw"

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE = sns.color_palette("muted")

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_RAW  : {DATA_RAW}")

## 1 · Load Raw Files

In [ ]:
files = {
    "transactions": "transaction_data.csv",
    "causal":       "causal_data.csv",
    "products":     "product.csv",
    "campaigns":    "campaign_desc.csv",
    "campaign_hh":  "campaign_table.csv",
    "coupons":      "coupon.csv",
    "redemptions":  "coupon_redempt.csv",
    "demographics": "hh_demographic.csv",
}

dfs = {name: pd.read_csv(DATA_RAW / fname) for name, fname in files.items()}

print(f"{'Dataset':<15} {'Rows':>10} {'Cols':>6}")
print("─" * 33)
for name, df in dfs.items():
    print(f"{name:<15} {len(df):>10,} {df.shape[1]:>6}")

## 2 · Schema Inspection

Dtypes, null rates, unique counts, and a sample value for every column in every file.

In [ ]:
for name, df in dfs.items():
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    schema = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "null_%":   null_pct,
        "n_unique": df.nunique(),
        "sample":   df.iloc[0] if len(df) else pd.Series(dtype=object),
    })
    print(f"\n{'='*60}")
    print(f"  {name.upper()}  — {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*60}")
    print(schema.to_string())

## 3 · Entity Counts

High-level inventory of unique entities across the study.

In [ ]:
tx   = dfs["transactions"]
prod = dfs["products"]

entities = {
    "Households":          tx["household_key"].nunique(),
    "Products (in tx)":    tx["PRODUCT_ID"].nunique(),
    "Products (catalog)":  prod["PRODUCT_ID"].nunique(),
    "Stores":              tx["STORE_ID"].nunique(),
    "Weeks":               tx["WEEK_NO"].nunique(),
    "Day range":           f"{tx['DAY'].min()} – {tx['DAY'].max()}",
    "Baskets":             tx["BASKET_ID"].nunique(),
    "Transaction lines":   len(tx),
    "Departments":         prod["DEPARTMENT"].nunique(),
    "Commodity descs":     prod["COMMODITY_DESC"].nunique(),
    "Sub-commodities":     prod["SUB_COMMODITY_DESC"].nunique(),
    "Campaigns":           dfs["campaigns"]["CAMPAIGN"].nunique(),
    "Unique coupons":      dfs["coupons"]["COUPON_UPC"].nunique(),
}

print(f"{'Entity':<26} {'Value':>12}")
print("─" * 40)
for k, v in entities.items():
    print(f"{k:<26} {str(v):>12}")

## 4 · Transaction Volume Over Time

Weekly basket and line-item counts across the study period.

In [ ]:
weekly = (
    tx.groupby("WEEK_NO")
      .agg(baskets=("BASKET_ID", "nunique"), lines=("PRODUCT_ID", "count"))
      .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(weekly["WEEK_NO"], weekly["baskets"], color=PALETTE[0], lw=1.5)
axes[0].set_ylabel("Unique Baskets")
axes[0].set_title("Weekly Basket Volume")

axes[1].plot(weekly["WEEK_NO"], weekly["lines"], color=PALETTE[1], lw=1.5)
axes[1].set_ylabel("Transaction Lines")
axes[1].set_xlabel("Week Number")
axes[1].set_title("Weekly Transaction Lines")

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

peak_week = weekly.loc[weekly["baskets"].idxmax(), "WEEK_NO"]
print(f"\nWeek range  : {weekly['WEEK_NO'].min()} – {weekly['WEEK_NO'].max()}")
print(f"Peak baskets: week {peak_week}  ({weekly['baskets'].max():,} unique baskets)")

## 5 · Pricing Coverage

The dataset does not contain a shelf-price column directly. We derive **implied unit price** as `SALES_VALUE / QUANTITY` and characterize the discount signals available for causal treatment assignment.

In [ ]:
# Implied unit price — exclude zero-quantity rows
tx_price = tx[tx["QUANTITY"] > 0].copy()
tx_price["unit_price"] = tx_price["SALES_VALUE"] / tx_price["QUANTITY"]

# RETAIL_DISC / COUPON_DISC are stored as negative values (discount off shelf price).
# Treat "active discount" as any non-zero value.
n = len(tx_price)
disc_stats = {
    "RETAIL_DISC != 0":       (tx_price["RETAIL_DISC"]       != 0).sum() / n,
    "COUPON_DISC != 0":       (tx_price["COUPON_DISC"]        != 0).sum() / n,
    "COUPON_MATCH_DISC != 0": (tx_price["COUPON_MATCH_DISC"]  != 0).sum() / n,
    "any discount":           ((tx_price[["RETAIL_DISC","COUPON_DISC","COUPON_MATCH_DISC"]] != 0).any(axis=1)).sum() / n,
}
print("Discount coverage (share of transaction lines):")
for k, v in disc_stats.items():
    print(f"  {k:<26} {v:6.1%}")

# Summary stats of retail_disc magnitude (non-zero only)
nz = tx_price["RETAIL_DISC"][tx_price["RETAIL_DISC"] != 0]
print(f"\nRETAIL_DISC magnitude (non-zero lines, stored as negative $):")
print(nz.describe(percentiles=[.05, .25, .5, .75, .95]).to_string())

# Summary stats for implied unit price
print("\nImplied unit price statistics:")
print(tx_price["unit_price"].describe(percentiles=[.05, .25, .5, .75, .95]).to_string())

In [ ]:
# Distribution of implied unit price by DEPARTMENT (top 10 by volume)
tx_dept = tx_price.merge(prod[["PRODUCT_ID", "DEPARTMENT"]], on="PRODUCT_ID", how="left")

top_depts = tx_dept["DEPARTMENT"].value_counts().nlargest(10).index
tx_top = tx_dept[tx_dept["DEPARTMENT"].isin(top_depts)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: log-scale price distribution per dept
for dept in top_depts:
    vals = tx_top.loc[tx_top["DEPARTMENT"] == dept, "unit_price"]
    vals = vals[(vals > 0) & (vals < vals.quantile(0.99))]
    axes[0].hist(np.log1p(vals), bins=60, alpha=0.4, label=dept, density=True)
axes[0].set_xlabel("log(1 + unit price)")
axes[0].set_title("Implied Unit Price Distribution by Dept (log scale)")
axes[0].legend(fontsize=7, ncol=2)

# Right: RETAIL_DISC magnitude distribution (non-zero; stored negative, plot as absolute value)
disc_vals = tx_price.loc[tx_price["RETAIL_DISC"] != 0, "RETAIL_DISC"].abs()
disc_vals = disc_vals[disc_vals < disc_vals.quantile(0.99)]
axes[1].hist(disc_vals, bins=60, color=PALETTE[3])
axes[1].set_xlabel("RETAIL_DISC magnitude ($)")
axes[1].set_title("Retail Discount Distribution\n(non-zero lines, absolute value)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

## 6 · Promotional Treatment Coverage

`causal_data.csv` captures store × product × week promotional exposure via `display` and `mailer` flags. These, together with `RETAIL_DISC` and coupons, form the available treatment signals for causal pricing estimation.

In [ ]:
causal = dfs["causal"].copy()

# display and mailer use string/mixed encoding: "0" = no promo, other values = promo type.
# pandas reads display as object/string; treat anything != "0" as active.
causal["display_str"] = causal["display"].astype(str).str.strip()
causal["mailer_str"]  = causal["mailer"].astype(str).str.strip()
causal["display_active"] = (causal["display_str"] != "0").astype(int)
causal["mailer_active"]  = (causal["mailer_str"]  != "0").astype(int)

display_active = causal["display_active"].astype(bool)
mailer_active  = causal["mailer_active"].astype(bool)
either = display_active | mailer_active
both   = display_active & mailer_active

# Flag coverage rates
n_causal = len(causal)
print("causal_data.csv — promotional flag coverage:")
print(f"  Total (product × store × week) obs : {n_causal:,}")
print(f"  display active (non-'0')            : {display_active.sum():,}  ({display_active.mean():.1%})")
print(f"  mailer  active (non-'0')            : {mailer_active.sum():,}  ({mailer_active.mean():.1%})")
print(f"  display OR mailer                   : {either.sum():,}  ({either.mean():.1%})")
print(f"  display AND mailer                  : {both.sum():,}  ({both.mean():.1%})")

print(f"\nDisplay code types:\n{causal['display_str'].value_counts().to_string()}")
print(f"\nMailer  code types:\n{causal['mailer_str'].value_counts().to_string()}")

print(f"\nUnique products with any promo exposure : {causal.loc[either, 'PRODUCT_ID'].nunique():,}")
print(f"Unique stores  in causal_data           : {causal['STORE_ID'].nunique():,}")
print(f"Unique weeks   in causal_data           : {causal['WEEK_NO'].nunique():,}")

In [ ]:
# Campaign and coupon reach
camp_hh  = dfs["campaign_hh"]
redempt  = dfs["redemptions"]

print("Campaign reach:")
print(f"  Unique campaigns              : {camp_hh['CAMPAIGN'].nunique():,}")
print(f"  Households in any campaign    : {camp_hh['household_key'].nunique():,}")
print(f"  Avg households per campaign   : {camp_hh.groupby('CAMPAIGN')['household_key'].nunique().mean():.0f}")

print("\nCoupon redemptions:")
print(f"  Total redemption events       : {len(redempt):,}")
print(f"  Unique households redeeming   : {redempt['household_key'].nunique():,}")
print(f"  Unique coupons redeemed       : {redempt['COUPON_UPC'].nunique():,}")
print(f"  Day range                     : {redempt['DAY'].min()} – {redempt['DAY'].max()}")

# Overlap: RETAIL_DISC active on same transaction as coupon-redeeming household
tx_with_hh = tx_price.merge(
    redempt[["household_key", "DAY"]].drop_duplicates(),
    on=["household_key", "DAY"], how="left", indicator=True
)
overlap = tx_with_hh[tx_with_hh["_merge"] == "both"]
print(f"\nLines where household redeemed coupon same day : {len(overlap):,}")
print(f"  Of which also had RETAIL_DISC != 0           : {(overlap['RETAIL_DISC'] != 0).sum():,} ({(overlap['RETAIL_DISC']!=0).mean():.1%})")

In [ ]:
# Heatmap: product treatment variation
# For each product, share of transactions with RETAIL_DISC active (non-zero),
# display active, mailer active, or any treatment — top 30 products by volume.

# Use the display_active / mailer_active flags computed above
causal_flags = causal[["PRODUCT_ID","STORE_ID","WEEK_NO","display_active","mailer_active"]]

tx_merged = tx_price.merge(causal_flags, on=["PRODUCT_ID","STORE_ID","WEEK_NO"], how="left")
tx_merged["display_active"] = tx_merged["display_active"].fillna(0)
tx_merged["mailer_active"]  = tx_merged["mailer_active"].fillna(0)
tx_merged["any_treatment"] = (
    (tx_merged["RETAIL_DISC"] != 0) |
    (tx_merged["display_active"] == 1) |
    (tx_merged["mailer_active"]  == 1)
).astype(int)

top30_products = tx_merged["PRODUCT_ID"].value_counts().nlargest(30).index
summary_trt = (
    tx_merged[tx_merged["PRODUCT_ID"].isin(top30_products)]
    .groupby("PRODUCT_ID")
    .agg(
        n_lines        =("PRODUCT_ID", "count"),
        pct_retail_disc=("RETAIL_DISC",       lambda x: (x != 0).mean()),
        pct_display    =("display_active",    "mean"),
        pct_mailer     =("mailer_active",     "mean"),
        pct_any        =("any_treatment",     "mean"),
    )
    .sort_values("n_lines", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 7))
heat_data = summary_trt[["pct_retail_disc","pct_display","pct_mailer","pct_any"]].rename(columns={
    "pct_retail_disc": "RETAIL_DISC",
    "pct_display":     "display",
    "pct_mailer":      "mailer",
    "pct_any":         "any treatment",
})
sns.heatmap(heat_data, annot=True, fmt=".0%", cmap="YlOrRd", ax=ax,
            linewidths=0.3, cbar_kws={"label": "Share of tx lines"})
ax.set_title("Treatment Exposure — Top 30 Products by Volume")
ax.set_xlabel("")
ax.set_ylabel("PRODUCT_ID")
plt.tight_layout()
plt.show()

print(f"\nProducts with any treatment on ≥20% of lines: "
      f"{(summary_trt['pct_any'] >= 0.20).sum()} of {len(summary_trt)}")

## 7 · Product Catalog Structure

Characterize the shape of the product hierarchy: DEPARTMENT → COMMODITY_DESC → SUB_COMMODITY_DESC → PRODUCT_ID.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# — 1: Products per DEPARTMENT (bar) —
dept_counts = prod["DEPARTMENT"].value_counts()
dept_counts.plot(kind="barh", ax=axes[0], color=PALETTE[0])
axes[0].invert_yaxis()
axes[0].set_xlabel("Unique Products")
axes[0].set_title("Products per Department")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# — 2: National vs Private brand split —
brand_counts = prod["BRAND"].value_counts()
axes[1].pie(brand_counts, labels=brand_counts.index, autopct="%1.1f%%",
            colors=PALETTE[:len(brand_counts)], startangle=90)
axes[1].set_title("Brand Type Mix")

# — 3: Products per COMMODITY_DESC — Zipf/rank plot —
ppc = prod.groupby("COMMODITY_DESC")["PRODUCT_ID"].nunique().sort_values(ascending=False)
ranks  = np.arange(1, len(ppc) + 1)
freq   = ppc.values
axes[2].scatter(np.log(ranks), np.log(freq), s=8, alpha=0.6, color=PALETTE[2])
axes[2].set_xlabel("log(Rank)")
axes[2].set_ylabel("log(Products)")
axes[2].set_title("Products per Commodity (Zipf plot)")

plt.tight_layout()
plt.show()

print(f"\nTop 10 commodities by product count:")
print(ppc.head(10).to_string())

## 8 · Field Gap Analysis

Map every available column to its role in the causal pricing framework and flag what is missing.

In [ ]:
gap_rows = [
    # (Field, Source file, Causal role, Notes)
    ("RETAIL_DISC",          "transaction_data", "Treatment",    "Retailer-applied price reduction; primary price treatment"),
    ("COUPON_DISC",          "transaction_data", "Treatment",    "Manufacturer coupon discount per line"),
    ("COUPON_MATCH_DISC",    "transaction_data", "Treatment",    "Retailer coupon-match discount per line"),
    ("display",              "causal_data",      "Treatment",    "In-store display flag (product × store × week)"),
    ("mailer",               "causal_data",      "Treatment",    "Direct-mail promotional flag (product × store × week)"),
    ("SALES_VALUE / QTY",    "derived",          "Treatment",    "Implied unit price proxy; noisy but week-level variation"),
    ("QUANTITY",             "transaction_data", "Outcome",      "Units purchased — primary demand outcome"),
    ("SALES_VALUE",          "transaction_data", "Outcome",      "Revenue; secondary outcome"),
    ("BASKET_ID",            "transaction_data", "Outcome",      "Basket membership — enables spillover / basket-level analysis"),
    ("CAMPAIGN",             "campaign_table",   "Instrument",   "Household-level campaign assignment; exogenous promo trigger"),
    ("COUPON_UPC",           "coupon / redempt", "Instrument",   "Coupon exposure + redemption; partial instrument for discount take-up"),
    ("START_DAY / END_DAY",  "campaign_desc",    "Instrument",   "Campaign timing — discontinuity around campaign window"),
    ("DEPARTMENT",           "product",          "Confounder",   "Category-level demand seasonality"),
    ("BRAND",                "product",          "Confounder",   "National vs. Private brand preference"),
    ("COMMODITY_DESC",       "product",          "Confounder",   "Fine-grained category; useful for graph edge construction"),
    ("household_key demos",  "hh_demographic",   "Confounder",   "Age/income/kids/homeowner — heterogeneous treatment response"),
    ("WEEK_NO / DAY",        "transaction_data", "Confounder",   "Seasonality, holidays"),
    ("STORE_ID",             "transaction_data", "Confounder",   "Store-level pricing / demand differences"),
    # --- Missing ---
    ("Shelf price",          "MISSING",          "❌ Missing",   "No pre-discount price; must be approximated or simulated"),
    ("Competitor prices",    "MISSING",          "❌ Missing",   "Out-of-store substitution effects unobservable"),
    ("Stock / inventory",    "MISSING",          "❌ Missing",   "Stockouts confound both discount and demand signals"),
    ("Unit cost / margin",   "MISSING",          "❌ Missing",   "Needed for profit-impact estimation"),
    ("Online / channel",     "MISSING",          "❌ Missing",   "Single-channel; cross-channel switching unobservable"),
]

gap_df = pd.DataFrame(gap_rows, columns=["Field", "Source", "Causal Role", "Notes"])

# colour-code the output
role_order = ["Treatment", "Outcome", "Instrument", "Confounder", "❌ Missing"]
gap_df["Causal Role"] = pd.Categorical(gap_df["Causal Role"], categories=role_order, ordered=True)
gap_df = gap_df.sort_values("Causal Role").reset_index(drop=True)

print(gap_df.to_string(index=False))

In [ ]:
# Quantify treatment variation at the product level — flag products
# that have enough treated observations for causal identification.
#
# Criterion: ≥5% and ≤95% of weekly observations show RETAIL_DISC != 0
# (sufficient variation, not always-on or always-off).

tx_wk = tx_price.groupby(["PRODUCT_ID", "WEEK_NO"]).agg(
    total_lines  =("PRODUCT_ID", "count"),
    disc_lines   =("RETAIL_DISC", lambda x: (x != 0).sum()),
).reset_index()
tx_wk["disc_rate"] = tx_wk["disc_lines"] / tx_wk["total_lines"]

product_variation = (
    tx_wk.groupby("PRODUCT_ID")["disc_rate"]
         .agg(mean_disc_rate="mean", weeks_observed="count")
         .reset_index()
)
product_variation["has_variation"] = (
    (product_variation["mean_disc_rate"] >= 0.05) &
    (product_variation["mean_disc_rate"] <= 0.95) &
    (product_variation["weeks_observed"] >= 10)
)

n_identifiable = product_variation["has_variation"].sum()
n_total = len(product_variation)
print(f"Products with identifiable RETAIL_DISC treatment variation:")
print(f"  {n_identifiable:,} / {n_total:,}  ({n_identifiable/n_total:.1%})")

# Distribution of mean discount rate
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(product_variation["mean_disc_rate"], bins=50, color=PALETTE[0], edgecolor="none")
ax.axvline(0.05, color="red",    ls="--", lw=1.2, label="5% threshold")
ax.axvline(0.95, color="orange", ls="--", lw=1.2, label="95% threshold")
ax.set_xlabel("Mean weekly RETAIL_DISC rate per product")
ax.set_ylabel("Products")
ax.set_title("Distribution of Treatment Variation Across Products")
ax.legend()
plt.tight_layout()
plt.show()

## 9 · Simulation Layer — Decision

**Decision: deferred to evaluation phase.**

The Complete Journey dataset contains genuine price variation through several observed channels:

| Signal | Type | Notes |
|---|---|---|
| `RETAIL_DISC` | Continuous | Retailer-applied discount; week × product variation |
| `display` / `mailer` | Binary | Store-level promo exposure |
| `COUPON_DISC` / `COUPON_MATCH_DISC` | Continuous | Coupon-driven price reductions |
| Campaign assignment | Instrument | Exogenous household-level exposure |

These real signals support causal identification without fabricating prices. The **one gap synthetic data fills** is _ground-truth elasticities_ for benchmarking causal estimators — real data cannot tell you the true underlying effect.

**Plan:**
- Notebooks 02–03 use the observed treatment signals directly.
- Notebook 04 / evaluation design: a **targeted semi-synthetic augmentation** layer will inject known price perturbations on top of observed prices to create a benchmark with ground-truth causal effects. This preserves the real co-purchase, catalog, and promotional structure while enabling causal recovery evaluation.